Enter API Key

In [ ]:
import openai
from getpass import getpass

api_key = getpass("Enter your OpenAI API key: ")

# initialize the OpenAI client with proxy settings
client = openai.OpenAI(
    api_key=api_key,
    base_url="https://ai-research-proxy.azurewebsites.net"  # LiteLLM Proxy
)


Enter your OpenAI API key: ··········


# Annotated testing dataset

#### Creating a random sample to annotate

In [ ]:
filtered_df = df[df["missing_key_info"] != 1]

total_rows = len(df)

filtered_rows = len(filtered_df)

print("Total rows in df:", total_rows)
print("Rows in filtered_df (missing_key_info != 1):", filtered_rows)

Total rows in df: 5642
Rows in filtered_df (missing_key_info != 1): 5062


In [ ]:
sample = filtered_df.sample(n=250, random_state=88)

n = len(sample)

# evenly split the rows across three annotators
annotator_names = ['Valeria', 'Zina', 'Marko']
annotators = [annotator_names[i % 3] for i in range(n)]

sample['annotator'] = annotators

sample = sample.sort_values(by="annotator").reset_index(drop=True)

sample["NACE code"] = ""
sample["Stance"] = ""
sample["Argument type(s)"] = ""

# number of comments per annotator
annotator_counts = sample["annotator"].value_counts()
print(annotator_counts)

sample.to_csv("sample.csv", index=False)

annotator
Valeria    84
Marko      83
Zina       83
Name: count, dtype: int64


#### Opening the annotated testing dataset

In [ ]:
import pandas as pd

file_path = "/content/Testing Dataset - Final.csv"
annotations_df = pd.read_csv(file_path)

#### Annotated testing summary

In [ ]:
argument_column = "Argument types"

# mapping from short codes to full argument labels
argument_map = {
    "1.1": "1.1. Polymer of low concern",
    "1.2": "1.2. Negligible toxicity",

    "2.1": "2.1. There is no alternative",
    "2.2": "2.2. Alternatives underperform",

    "3.1": "3.1. Negative economic impact",
    "3.2": "3.2. Forced relocation",
    "3.3": "3.3. Job loss",
    "3.4": "3.4. Competitiveness loss",

    "4.1": "4.1. Impact on convenience",
    "4.2": "4.2. Impact on health and safety",
    "4.3": "4.3. Green transition",

    "5":   "5. Unclear/no arguments"
}

# dummy columns initialized to 0
for full_label in argument_map.values():
    annotations_df[full_label] = 0

# set dummy columns based on codes in "Argument types"
def assign_argument_types(row):
    if pd.notna(row[argument_column]):
        codes = [code.strip() for code in str(row[argument_column]).split(",")]
        for code in codes:
            if code in argument_map:
                full_label = argument_map[code]
                row[full_label] = 1
    return row

# apply the function row-wise
annotations_df = annotations_df.apply(assign_argument_types, axis=1)


In [ ]:
# count each argument type
argument_counts = annotations_df[list(argument_map.values())].sum().sort_values(ascending=False)

# display the counts
print(argument_counts)


5. Unclear/no arguments             56
2.1. There is no alternative        32
3.1. Negative economic impact       27
2.2. Alternatives underperform      18
4.2. Impact on health and safety    14
4.1. Impact on convenience          12
3.3. Job loss                       11
3.4. Competitiveness loss           10
1.1. Polymer of low concern          9
1.2. Negligible toxicity             9
4.3. Green transition                6
3.2. Forced relocation               3
dtype: int64


this one tries to avoid implicit argumentation

# Lobbying Argument Classification

## The Prompts

#### Standard prompt

Used by

*   Model 1 (the baseline): Zero-shot GPT-4o-mini (8B parameters)
*   Model 2: Zero-shot GPT-4o (300B parameters)
*   Model 6: Three-run ensemble (GPT-4o-mini)
*   Model 7: Five-run ensemble (GPT-4o-mini)
*   Model 8: Three-run ensemble (GPT-4o)



In [ ]:
prompt = f"""
You are an expert assistant classifying stakeholder comments related to the EU’s proposed PFAS ban. Your goal is to determine whether a given comment contains one or more lobbying arguments against the PFAS restriction and to identify their type based on the categories below.

Here are the instructions:

If the comment falls under one of the categories, output the number of the subargument.

If the prompt falls under multiple categories, output all of them, separated by commas (e.g., "1.1, 2.1, 3.4").

If no lobbying argument is clearly present, output only "5" (which is exclusive and cannot be combined with others).

Do not give extra reasoning.

Here are the possible lobbying argument categories:

    Scientific arguments
        1.1 Not all PFAS are dangerous - the argument cites polymers of low concern (PLC).
        1.2 A given PFAS is claimed to be not toxic.

    No alternative arguments
        2.1 There is no alternative to a given PFAS.
        2.2 Existing alternatives perform worse than PFAS-based materials

    Economic arguments
        3.1 Negative economic impact on business from the pfas restriction
        3.2 Production is forced to move out of Europe
        3.3 Production job loss because of the restrictions
        3.4 Loss of European (EU) competitiveness and autonomy (not company specific).

    Social arguments
        4.1 Negative impact on customers, convenience, and the modern life.
        4.2 Negative impact on the health and safety of citizens after the loss of PFAS-based products.
        4.3 PFAS are needed to achieve Green transition because of their efficiency to minimise emissions (Green Deal argument).

        5. Unclear or no arguments

Only output the label if the argument is clearly present in the comment. Minor paraphrasing or technical synonyms are allowed if the meaning directly aligns with a subcategory.

Now label the following comment:

{original_text}
"""

#### Chain of Thought prompt (Model 4)

In [ ]:
prompt = f"""
You are an expert assistant classifying stakeholder comments related to the EU’s proposed PFAS ban. Your goal is to determine whether a given comment contains one or more lobbying arguments against the PFAS restriction and to identify their type based on the categories below.

As you perform this task, follow these reasoning steps:

First: Understand the content and purpose of the comment. Determine whether it expresses opposition to the PFAS ban by presenting one or more specific arguments.

Second: Make a preliminary decision about which lobbying argument categories the comment falls into, if any.

Third: Reflect critically on your initial selection: Are the arguments clearly stated or merely implied? Only choose explicit references to arguments.

Fourth: Confirm your final classification by selecting only those subcategories where the argument is explicit and matches the definitions provided.

Then, output only the labels of the subcategories, separated by commas (e.g., "1.1, 2.1, 3.4").
If no lobbying argument is clearly present, output only "5" (which is exclusive and cannot be combined with others).

Here are the possible lobbying argument categories:

    Scientific arguments
        1.1 Not all PFAS are dangerous - the argument cites polymers of low concern (PLC).
        1.2 A given PFAS is claimed to be not toxic.

    No alternative arguments
        2.1 There is no alternative to a given PFAS.
        2.2 Existing alternatives perform worse than PFAS-based materials

    Economic arguments
        3.1 Negative economic impact on business from the pfas restriction
        3.2 Production is forced to move out of Europe
        3.3 Production job loss because of the restrictions
        3.4 Loss of European (EU) competitiveness and autonomy (not company specific).

    Social arguments
        4.1 Negative impact on customers, convenience, and the modern life.
        4.2 Negative impact on the health and safety of citizens after the loss of PFAS-based products.
        4.3 PFAS are needed to achieve Green transition because of their efficiency to minimise emissions (Green Deal argument).

        5. Unclear or no arguments

Only output the label if the argument is clearly present in the comment. Minor paraphrasing or technical synonyms are allowed if the meaning directly aligns with a subcategory.

Now label the following prompt:

{original_text}
"""

#### Prompt for Model 5 (Few-Shot)

See other notebook for exemplar selection

In [ ]:
prompt = f"""
You are an expert assistant classifying stakeholder comments related to the EU’s proposed PFAS ban. Your goal is to determine whether a given comment contains one or more lobbying arguments against the PFAS restriction and to identify their type based on the categories below.

Here are the instructions:

If the comment falls under one of the categories, output the number of the subargument.

If the prompt falls under multiple categories, output all of them, separated by commas (e.g., "1.1, 2.1, 3.4").

If no lobbying argument is clearly present, output only "5" (which is exclusive and cannot be combined with others).

Do not give extra reasoning.

Here are the possible lobbying argument categories:

    Scientific arguments
        1.1 Not all PFAS are dangerous - the argument cites polymers of low concern (PLC).
        1.2 A given PFAS is claimed to be not toxic.

    No alternative arguments
        2.1 There is no alternative to a given PFAS.
        2.2 Existing alternatives perform worse than PFAS-based materials

    Economic arguments
        3.1 Negative economic impact on business from the pfas restriction
        3.2 Production is forced to move out of Europe
        3.3 Production job loss because of the restrictions
        3.4 Loss of European (EU) competitiveness and autonomy (not company specific).

    Social arguments
        4.1 Negative impact on customers, convenience, and the modern life.
        4.2 Negative impact on the health and safety of citizens after the loss of PFAS-based products.
        4.3 PFAS are needed to achieve Green transition because of their efficiency to minimise emissions (Green Deal argument).

        5. Unclear or no arguments

Only output the label if the argument is clearly present in the comment. Minor paraphrasing or technical synonyms are allowed if the meaning directly aligns with a subcategory.

Here are some demonstrations of how to perform the task.

Example 1:

- Comment 1 "General comments: First, we endorse FCJ's statement on the issues of the proposed restriction, as attached in Section IV.  We manufacture and distribute PFPE lubricants. PFPE is considered a polymer of low concern by OECD standards and has no effect on the human body. In addition, PFPE is used even though it is more expensive than conventional lubricants because it has the following performance characteristics that only PFPE can provide. Non-volatile: Because it is an organofluorine compound, it has a high specific gravity (d = about 1.9), and even low-viscosity oils do not volatilize more than hydrocarbon oils. Thermal stability: It does not decompose even at high temperatures above 350°C. PFPE has high heat resistance and WO87/02995 states that it finally decomposes into low molecular weight oil at 500°C. Even silicone oil, which is said to have high heat resistance, has heat resistance up to 250°C. Post the URL of the silicone manufacturer's oil operating temperature. https://nihon-resin.jp/products/oil/kf-96/ Chemical Stability: PFPE oil is resistant to strong acids and alkalis due to its CF bond stability. Other oils are fragile due to their chemical structure, causing chemical structure changes and decomposition due to strong acids, alkalis, etc. In addition, since it does not dissolve plastics, it is the only oil that can be used to lubricate plastics. In this way, PFPE is used because of the properties that result from its chemical structure, and oils with different structures cannot produce the same properties and cannot be substituted. For these reasons, PFPE-based lubricants are not appropriate within the proposed 12-year period and should be exempt from regulation. 6937 1. Sectors and (sub-)uses: Lubricants 6937 3. Emissions in the end-of-life phase: In general, equipment and bearings that use lubricating oil and grease are recycled as steel scrap. When it is recycled, it is melted in an electric furnace. The melting temperature is 1,600 ° C to 1,800 ° C, the gas emitted from it is 1,500 ° C, and all organic matter is decomposed into inorganic gas. https://worldwide.espacenet.com/patent/search/family/040965236/publication/JP5362983B2?q=pn%3DJP5362983B2 It is judged that in an advanced society where recycling is established, it will not accumulate in the environment as is a concern in the proposed regulation. If we are concerned about the safety of chemical substances, we believe that we should build a recycling system that can incinerate them at an appropriate temperature. 6937 8. Other identified uses: To be described in section V."

- Output 1: "1.1, 1.2, 2.2"


Example 2:

- Comment 2 "6. Missing uses: a. The annual tonnage and emissions (at sub-sector level) and type of PFAS associated with the relevant use. Wet chemical production equipment for GreenEnergy (Silicon Photovoltaics), Silicon Prime Wafering, Semiconductor Chips, Medtech and Glass Display require machine design using resistant materials to avoid degradation and corrosion and thus pollution of environment and goods produced during their life time.  While the total value of PFAS in RENA: ETFE – ECTFE – FEP – FEPM – FFKM – FKM - PCTFE – PFA – PTFE – PVDF sum up to total purchase volume of ~20Mio€ an calculated annual mass (*specific bulk PFAS price + 25% cost surplus for shaping material as needed) above 250 tons is shipped in RENA products. Including companies over entire EU the sub-sector (wet chemical equipment)  is estimated to reach 800 tons. This will be overshot by far when chemical facility installations (not part of the subs-sector here) get included Thus materials emission occurs during manufacturing due to milling and drilling. These shaving residues get collected for appropriate thermal disposal (halide exhaust scrubber) and are expected to reach 2% of total PFAS tonnage which sums up to approximately 5 metric tons which get not emitted to environment. Due to the quality and industrial use life cycle of wet chemical equipment ranges from 10 to over 20 years with an estimated average value of 15 years. During life time amounts PFAS concentrations emitted vial process chemicals and water are below current analytical capacities and in our opinion negligible. While emission of PFAS particles is extremely low during usage post life time dismantling of equipment has highest importance. Regarding subsequent required waste treatment a thermal process including appropriate exhaust treatment due fluorine contents is necessary.  However, due to current regulations the responsibility for dismantling and appropriate disposal of PFAS parts is solely with the owner and has to be done according to the local authority regulations. Thus the final disposal is today out of responsibility of the equipment manufacturer.  b. The key functionalities provided by PFAS for the relevant use. Due to harsh chemicals used in the field of etching, cleaning and coating only materials are qualified which do not create emission by chemical bleed-out or plastic particles creation. In addition these PFAS materials used to avoid accidental chemical leakages and hazards to the people and environment. Thus the application of highly resistant material is mandatory to achieve the needed safety and cleanliness to produce state-of-the-art semiconductor grade wafers, semiconductor chips, silicon solar cells and medtech products.  c. The number of companies in the sector estimated to be affected by the restriction. The total number of active suppliers in the wet chemcial equipment business is estimated 20 companies in Europa with a turnover of approximately 1 Billion € which facing much bigger competitors from Japan, Korea, USA and China with a total world market of estimated 20 Billion €.  d. The availability, technical and economic feasibility, hazards and risks of alternatives for the relevant use, including information on the extent (in terms of market shares) to which alternative-based products are already offered on the EU market and whether any shortages in the supply of relevant alternatives are expected. Due to the factor 5-10 higher pricing of PFAS material alternatives are used wherever possible due to this highly competitive market environment. In the case of general restriction investments for semiconductor, photovoltaics and other related electronic fabs would suffer not proper functioning of wet chemical equipment. Resulting accidental leakages and function failure will lead to long down time and hardly predictable maintenance in such facilities with the consequence of low yields and poor qualitative output of goods. Such an investment will most probably not reach a competitive level compared to fabs outside Europe without PFAS restriction.  e. For cases in which alternatives are not yet available, information on the status of R&D processes for finding suitable alternatives, including the extent of R&D initiatives in terms of time and/or financial investments, the likelihood of successful completion, the time expected to be required for substitution (including any relevant certification or regulatory approvals) and the major challenges encountered with alternatives which were considered but subsequently disregarded. Harsh and highly hazardous chemicals in combinations: e.g. Fluoric Acid, Nitric Acid, Sulfuric Acid, Peroxide, Ozone, Chlorine and derivates, Sodium and Potassium Hydroxide at elevated temperatures (above 120°C) are standard and require safe handling in clean room environments. For these applications no alternative PFAS materials are known. General material testing to achieve qualification for chemical process noted above reach a level of 15% of entire R&D budget while 90% of since today identified and qualified materials are listed as PFAS according to ANNEX XV RESTRICTION REPORT.  f.  no information on this topic  g. For cases in which substitution is not technically or economically feasible, information on what the socio-economic impacts would be for companies, consumers, and other affected actors. If available, please provide the annual value of EU sales and profits of the relevant sector, and employment numbers for the sector. About 90% of sector´s products would not achieve proper functioning (density, corrosion, contamination of produced good) which would be a considerable disadvantage compared to non-EU competitors. As consequence a reduction of turnover from ~1.000Mio€ down to 100Mio€ can be expected. Such fundamental restructuring would mean at least in the same share a reduction of employments from 4.000 down to 800. This shrinkage is estimated to be under critical in many cases to supply to international semiconductor and photovoltaics companies, thus a transfer of companies productions to countries outside Europe would be a most probable measure to maintain as a suppliers in this world wide businesses where PFAS material usage  is of high importance."

- Output 2: "2.1, 2.2, 3.1, 3.2, 3.3, 3.4"

Example 3:

- Comment 3 "I hope that the bill will go through so we can know that our children can grow up without PFAS in products and drinking water. It should not be a part of our children's everyday life. I am prepared to live with products with impaired properties, as this bill may have positive effects for several generations to come."

- Output 3: "5"

Example 4:

- Comment 4 "General comments: General comments are attached in Section Ⅳ. 4631 1. Sectors and (sub-)uses: Our product application is fishing line made from PVDF, so we assume Consumer Mixture is applicable as sector listed in in the Annex XV restriction report (Table 9), but nothing is listed as 8sub-) uses. 4631 2. Emissions in the end-of-life phase: Please see the Confidential Attachment in Section V. 4631 3. Emissions in the end-of-life phase: Please see the Confidential Attachment in Section V. 4631 4. Impacts on the recycling industry: Included in the general comment attached in Section IV. 4631 5. Proposed derogations: Please see the Confidential Attachment in Section V. 4631 6. Missing uses: Please see the Confidential Attachment in Section V. 4631 7. Potential derogations marked for reconsideration: Please see the Confidential Attachment in Section V. 4631 8. Other identified uses: Please see the Confidential Attachment in Section V. 4631 9. Degradation potential of specific PFAS sub-groups: Please see the Confidential Attachment in Section V. 4631 10. Analytical methods: Please see the Confidential Attachment in Section V."

- Output 4: "5"

Now label the following comment:

{original_text}
"""

## Classification code

In [ ]:
# open datasets

import pandas as pd

file_path = "/content/Testing Dataset - Final.csv"
annotations_df = pd.read_csv(file_path)

file_path = "/content/comments_definitive.csv"
df_full = pd.read_csv(file_path)


***Prompt*** variable chosen as one of the prompts listed above. Currently selected: ***standard prompt***.

In [ ]:
import pandas as pd
import time
from tqdm import tqdm
import openai

responses = {}
start_time = time.time()

for _, row in tqdm(df_full.iterrows(), total=len(df_full), desc="Classifying"):
    comment_id = row["ID"]
    original_text = row["Combined Comments"]

    if pd.isna(original_text) or not str(original_text).strip():
        responses[comment_id] = "5"
        continue

    prompt = argument_classification_prompt = f"""
You are an expert assistant classifying stakeholder comments related to the EU’s proposed PFAS ban. Your goal is to determine whether a given comment contains one or more lobbying arguments against the PFAS restriction and to identify their type based on the categories below.

Here are the instructions:

If the comment falls under one of the categories, output the number of the subargument.

If the prompt falls under multiple categories, output all of them, separated by commas (e.g., "1.1, 2.1, 3.4").

If no lobbying argument is clearly present, output only "5" (which is exclusive and cannot be combined with others).

Do not give extra reasoning.

Here are the possible lobbying argument categories:

    Scientific arguments
        1.1 Not all PFAS are dangerous - the argument cites polymers of low concern (PLC).
        1.2 A given PFAS is claimed to be not toxic.

    No alternative arguments
        2.1 There is no alternative to a given PFAS.
        2.2 Existing alternatives perform worse than PFAS-based materials

    Economic arguments
        3.1 Negative economic impact on business from the pfas restriction
        3.2 Production is forced to move out of Europe
        3.3 Production job loss because of the restrictions
        3.4 Loss of European (EU) competitiveness and autonomy (not company specific).

    Social arguments
        4.1 Negative impact on customers, convenience, and the modern life.
        4.2 Negative impact on the health and safety of citizens after the loss of PFAS-based products.
        4.3 PFAS are needed to achieve Green transition because of their efficiency to minimise emissions (Green Deal argument).

        5. Unclear or no arguments

Only output the label if the argument is clearly present in the comment. Minor paraphrasing or technical synonyms are allowed if the meaning directly aligns with a subcategory.

Now label the following comment:

{original_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        label = response.choices[0].message.content.strip()
        responses[comment_id] = label
    except Exception as e:
        print(f"Error on {comment_id}: {e}")
        responses[comment_id] = "ERROR"

elapsed = time.time()
print(f"Finished in {elapsed - start_time:.2f} seconds.")



Classifying: 100%|██████████| 5642/5642 [50:53<00:00,  1.85it/s]

Finished in 3053.15 seconds.


In [ ]:
print(responses)

{3834: '5', 3843: '1.1, 2.1, 3.4, 4.3', 3849: '5', 3850: '5', 3851: '5', 3852: '2.1, 3.1, 3.4, 4.1', 3853: '2.1, 3.1, 3.4', 3854: '5', 3855: '1.1, 1.2, 3.4, 4.3', 3856: '2.1, 3.1, 3.4, 4.1', 3858: '2.1, 3.1, 3.4', 3860: '2.1, 3.1', 3861: '2.1', 3862: '5', 3863: '5', 3864: '2.1, 2.2', 3865: '1.1, 2.1, 3.1, 3.4, 4.3', 3866: '2.1, 3.1, 3.4', 3867: '2.1, 3.2, 3.3, 3.4', 3868: '2.1, 3.1', 3869: '1.1, 2.1, 3.4', 3870: '5', 3871: '2.1', 3872: '2.1', 3873: '2.1, 3.1, 3.3', 3875: '2.1, 3.1, 3.4', 3876: '1.1, 2.1, 3.1, 3.3', 3877: '2.1', 3878: '2.1, 3.1, 3.3, 4.1', 3879: '5', 3880: '5', 3881: '2.1, 3.1, 3.3', 3882: '2.1', 3883: '2.1, 3.1, 3.4, 4.3', 3884: '5', 3885: '2.1, 3.1, 3.2, 3.3, 3.4', 3886: '1.1, 2.1, 3.2', 3887: '2.1, 3.1, 3.3, 3.4, 4.1, 4.2, 4.3', 3888: '2.1, 3.1, 3.4', 3889: '2.1, 3.1, 3.4, 4.2', 3890: '2.1, 3.1, 3.4', 3891: '2.1, 3.1, 3.3, 3.4, 4.1, 4.3', 3892: '2.1', 3893: '2.1', 3894: '1.1, 2.1, 3.1, 3.4, 4.2', 3895: '5', 3896: '2.1, 3.1, 3.4', 3897: '1.1, 2.1, 3.1, 3.4, 4.1', 3898

Saving and organising classifications

In [ ]:
# convert to dataframe and dummify
response_df = pd.DataFrame.from_dict(responses, orient='index', columns=["Response"]).reset_index().rename(columns={"index": "ID"})
annotations_df = annotations_df.drop(columns=["Response"], errors="ignore")
annotations_df = annotations_df.merge(response_df, on="ID")
annotations_df["Response"] = annotations_df["Response"].fillna("5").str.replace(" ", "")
df_dummies = annotations_df["Response"].str.get_dummies(sep=",")
df_dummies.columns = [f"{col}_pred" for col in df_dummies.columns]
df_final = pd.concat([annotations_df, df_dummies], axis=1)

pred_columns = [col for col in df_final.columns if col.endswith("_pred")]

other_pred_columns = [col for col in pred_columns if col != "5_pred"]

df_final.loc[df_final[other_pred_columns].any(axis=1), '5_pred'] = 0

# saving results
#df_final.to_csv("pfas_classification_results.csv", index=False)

Evaluating against the ground truth

#### Argument Premises

In [ ]:
df_final["Argument types"] = df_final["Argument types"].fillna("5").astype(str).str.replace(" ", "")

df_eval_dummies = df_final["Argument types"].str.get_dummies(sep=",")

df_eval_dummies = df_eval_dummies.rename(columns={col: f"{col}_eval" for col in df_eval_dummies.columns})

df_combined = pd.concat([df_final, df_eval_dummies], axis=1)

In [ ]:
# extract true labels
y_true = df_combined.filter(regex=r'^(\d\.\d|5)_eval$')
y_true = y_true.drop(columns=[col for col in ["1.3_eval", "1.4_eval"] if col in y_true.columns])

# extract predicted labels
y_pred = df_combined.filter(regex=r'^(\d\.\d|5)_pred$')

expected_pred_columns = [
    "1.1_pred", "1.2_pred", "2.1_pred", "2.2_pred", "3.1_pred", "3.2_pred", "3.3_pred", "3.4_pred",
    "4.1_pred", "4.2_pred", "4.3_pred", "5_pred"
]

for col in expected_pred_columns:
    if col not in y_pred.columns:
        y_pred[col] = 0

y_true = y_true.sort_index(axis=1)
y_pred = y_pred.sort_index(axis=1)

#### Argument Categories

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=y_true.columns))

In [ ]:
import pandas as pd

responses = {}

category_mapping = {
    "1": ["1.1", "1.2"],
    "2": ["2.1", "2.2"],
    "3": ["3.1", "3.2", "3.3", "3.4"],
    "4": ["4.1", "4.2", "4.3"],
    "5": ["5"],
}

def responses_to_category_df(resp_dict):
    """
    Returns a DF where each row → one index in `resp_dict`
    and each column (1 … 5) is 1 if that category appears, else 0.
    """
    rows = []
    for idx, label_str in resp_dict.items():
        # normalise the string to a list of trimmed tokens
        subs = [tok.strip() for tok in label_str.split(",")]
        row = {"index": idx}
        for cat, subcats in category_mapping.items():
            row[cat] = int(any(sc in subs for sc in subcats))
        rows.append(row)
    df = pd.DataFrame(rows).set_index("index").sort_index()
    return df

cat_df = responses_to_category_df(responses)

summary = (
    cat_df.sum()                                  # raw count per category
          .to_frame(name="count")
          .assign(percent=lambda d: 100*d["count"] / len(cat_df))
          .round(2)
)

print("Prevalence of top-level categories (n = {:,})".format(len(cat_df)))
print(summary)



#### Code for Result aggregations (Ensemble models)

In [ ]:
from collections import defaultdict

# run1, run2, etc. are the resulting dictionaries from each run

run1 = {5057: '5', 6661: '2.2, 3.1', 6927: '2.1, 2.2', 6396: '5', 5243: '5', 6079: '5', 5867: '5', 7324: '2.1, 2.2, 3.1, 3.4', 4073: '5', 4879: '5', 5008: '5', 6245: '5', 6742: '2.1, 2.2', 4465: '1.1, 2.1, 2.2, 4.2', 6578: '2.1, 2.2, 3.1', 5848: '5', 5915: '5', 8710: '1.1, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.3', 9292: '5', 6805: '2.1, 2.2, 3.1, 3.3', 7281: '1.1, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2', 8728: '3.1, 3.3', 8333: '2.1, 2.2, 4.3', 4558: '2.1, 4.1', 7232: '1.1, 1.2, 2.1, 2.2, 3.1, 3.3, 4.1, 4.2', 8894: '1.1, 2.1, 3.1', 4240: '5', 6425: '4.1, 4.3', 7435: '2.1, 2.2, 3.1, 4.1', 9318: '2.2', 8968: '2.1, 2.2', 5812: '5', 6072: '5', 6290: '5', 7827: '2.1, 2.2', 4258: '2.1, 2.2, 3.4', 5944: '5', 8397: '1.1, 1.2, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2', 8641: '2.1, 2.2', 9218: '5', 7226: '2.2', 8233: '2.1, 4.2', 8777: '2.1, 2.2, 3.1, 3.3, 4.2', 5368: '5', 5031: '5', 8611: '1.1', 9552: '2.1, 3.1', 9126: '5', 8196: '1.1', 7959: '2.1, 2.2, 3.1, 4.3', 4859: '5', 4356: '1.1, 2.1, 4.2', 4605: '5', 8337: '5', 5215: '5', 5049: '5', 7663: '5', 4738: '5', 9246: '2.2', 4932: '5', 7537: '1.1, 2.1, 3.1', 5249: '5', 4065: '5', 6685: '1.1, 2.1, 3.2, 3.3', 7298: '2.1, 2.2, 3.1', 4166: '2.1, 2.2', 4983: '5', 9045: '2.1, 2.2, 3.1, 3.2', 5579: '5', 7980: '2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1', 8663: '5', 8345: '2.1, 2.2, 3.1, 3.3, 3.4, 4.2, 4.3', 8961: '4.2', 7204: '5', 8465: '5', 6015: '5', 5451: '5', 7532: '1.1, 2.1, 2.2, 3.1, 3.3, 3.4', 4405: '5', 7146: '5', 9585: '3.4', 9035: '2.1, 4.2', 7033: '2.1, 2.2, 3.1, 3.3', 4813: '5', 6401: '5', 4211: '5', 5199: '5', 4516: '5', 9150: '1.1, 2.1, 2.2, 3.1, 3.3, 3.4', 8764: '5', 8561: '5', 5300: '5', 8255: '2.2', 7191: '5', 5427: '5', 5023: '5', 7735: '5', 6058: '1.1, 2.1, 2.2, 3.1, 3.3, 3.4, 4.2, 4.3', 4277: '5', 7726: '5'}
run2 = {5057: '5', 6661: '2.1, 3.1, 3.4', 6927: '2.1, 2.2', 6396: '5', 5243: '5', 6079: '5', 5867: '5', 7324: '2.1, 2.2, 3.1, 3.4', 4073: '5', 4879: '5', 5008: '5', 6245: '5', 6742: '2.1, 2.2', 4465: '1.1, 2.1, 2.2, 4.2', 6578: '2.1, 2.2, 3.1', 5848: '5', 5915: '5', 8710: '1.1, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.3', 9292: '5', 6805: '2.1, 2.2, 3.1, 3.3', 7281: '1.1, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2', 8728: '3.1, 3.3', 8333: '2.1, 2.2, 3.4', 4558: '2.1, 4.1', 7232: '1.1, 1.2, 2.1, 2.2, 3.1, 3.3, 4.1, 4.2', 8894: '1.1, 2.1, 3.1', 4240: '5', 6425: '4.1, 4.3', 7435: '2.1, 2.2, 3.1, 4.1', 9318: '2.2', 8968: '2.1, 2.2', 5812: '5', 6072: '5', 6290: '5', 7827: '2.1, 2.2', 4258: '2.1, 2.2, 3.1, 3.4', 5944: '5', 8397: '1.1, 1.2, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2', 8641: '2.1', 9218: '5', 7226: '2.2', 8233: '2.1, 4.2', 8777: '2.1, 2.2, 3.1, 3.3, 4.2', 5368: '5', 5031: '5', 8611: '1.1', 9552: '2.1, 3.1', 9126: '5', 8196: '1.1', 7959: '2.1, 2.2, 3.1, 4.3', 4859: '5', 4356: '1.1, 2.1, 4.2', 4605: '5', 8337: '5', 5215: '5', 5049: '5', 7663: '5', 4738: '5', 9246: '2.2', 4932: '5', 7537: '1.1, 2.1, 3.1, 4.2', 5249: '5', 4065: '5', 6685: '1.1, 2.1, 3.2, 3.3', 7298: '2.1, 2.2, 3.1', 4166: '2.1', 4983: '5', 9045: '2.1, 2.2, 3.1, 3.2', 5579: '5', 7980: '2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1', 8663: '5', 8345: '2.1, 2.2, 3.1, 3.3, 3.4, 4.2, 4.3', 8961: '4.2', 7204: '5', 8465: '5', 6015: '5', 5451: '5', 7532: '1.1, 2.1, 2.2, 3.1, 3.3, 3.4', 4405: '5', 7146: '5', 9585: '3.4', 9035: '2.1, 4.2', 7033: '1.1, 2.1, 2.2, 3.1, 3.3', 4813: '5', 6401: '5', 4211: '5', 5199: '5', 4516: '5', 9150: '1.1, 2.1, 2.2, 3.1, 3.3, 3.4', 8764: '5', 8561: '5', 5300: '5', 8255: '2.1, 2.2', 7191: '5', 5427: '5', 5023: '5', 7735: '5', 6058: '1.1, 2.1, 2.2, 3.1, 3.3, 3.4, 4.1, 4.2, 4.3', 4277: '5', 7726: '5'}
run3 = {5057: '5', 6661: '2.2, 3.1, 3.4', 6927: '2.1, 2.2', 6396: '5', 5243: '5', 6079: '5', 5867: '5', 7324: '2.1, 2.2, 3.1, 3.4', 4073: '5', 4879: '5', 5008: '5', 6245: '5', 6742: '2.1, 2.2, 3.1', 4465: '1.1, 2.1, 2.2, 4.2', 6578: '2.1, 2.2, 3.1', 5848: '5', 5915: '5', 8710: '1.1, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.3', 9292: '5', 6805: '2.1, 2.2, 3.1, 3.3', 7281: '1.1, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2', 8728: '3.1, 3.3', 8333: '2.1, 2.2, 3.4', 4558: '2.1, 4.1', 7232: '1.1, 1.2, 2.1, 2.2, 3.1, 3.3, 4.1, 4.2', 8894: '1.1, 2.1, 3.1', 4240: '5', 6425: '4.1, 4.3', 7435: '2.1, 2.2, 3.1, 4.1', 9318: '2.2', 8968: '2.1', 5812: '5', 6072: '5', 6290: '5', 7827: '2.1, 2.2', 4258: '2.1, 2.2, 3.4', 5944: '5', 8397: '1.1, 1.2, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2', 8641: '2.1', 9218: '5', 7226: '2.2', 8233: '2.1, 4.2', 8777: '2.1, 2.2, 3.1, 3.3, 4.2', 5368: '5', 5031: '5', 8611: '1.1', 9552: '2.1, 3.1', 9126: '5', 8196: '1.1', 7959: '2.1, 2.2, 3.1, 4.3', 4859: '5', 4356: '1.1, 2.1, 2.2, 4.2', 4605: '5', 8337: '5', 5215: '5', 5049: '5', 7663: '5', 4738: '5', 9246: '2.2, 3.4', 4932: '5', 7537: '1.1, 2.1, 3.1, 3.4', 5249: '5', 4065: '5', 6685: '1.1, 1.2, 2.1, 3.2, 3.3', 7298: '2.1, 2.2, 3.1', 4166: '2.1, 2.2', 4983: '5', 9045: '2.1, 2.2, 3.1, 3.2, 3.3', 5579: '5', 7980: '2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1', 8663: '5', 8345: '2.1, 2.2, 3.1, 3.3, 3.4, 4.2', 8961: '4.2', 7204: '5', 8465: '5', 6015: '5', 5451: '5', 7532: '1.1, 2.1, 2.2, 3.1, 3.3, 3.4', 4405: '5', 7146: '5', 9585: '3.4', 9035: '2.1, 4.2', 7033: '1.1, 2.1, 2.2, 3.1, 3.3', 4813: '5', 6401: '5', 4211: '5', 5199: '5', 4516: '5', 9150: '1.1, 2.1, 2.2, 3.1, 3.3, 3.4', 8764: '2.1', 8561: '5', 5300: '5', 8255: '2.1, 2.2', 7191: '5', 5427: '5', 5023: '5', 7735: '5', 6058: '1.1, 2.1, 2.2, 3.1, 3.3, 3.4, 4.2, 4.3', 4277: '5', 7726: '5'}

# In case of five runs, insert the additional results.
#run4 =
#run5 =

all_runs = [run1, run2, run3]

classification_counts = defaultdict(lambda: defaultdict(int))

for run in all_runs:
    for id, classifications_str in run.items():
        classifications = [c.strip() for c in classifications_str.split(',') if c.strip()]
        for classification in classifications:
            classification_counts[id][classification] += 1

# new dictionary with aggregated classification results
result_dict = {}

for id, counts in classification_counts.items():
    common_classifications = []
    for classification, count in counts.items():
        if count >= 2:
            common_classifications.append(classification)

    if common_classifications:
        result_dict[id] = ', '.join(sorted(common_classifications))
    elif 5 in counts and counts[5] >= 2:
         result_dict[id] = '5'


# resulting dictionary can now be used in the model evlauation code above
print(result_dict)

{5057: '5', 6661: '2.2, 3.1, 3.4', 6927: '2.1, 2.2', 6396: '5', 5243: '5', 6079: '5', 5867: '5', 7324: '2.1, 2.2, 3.1, 3.4', 4073: '5', 4879: '5', 5008: '5', 6245: '5', 6742: '2.1, 2.2', 4465: '1.1, 2.1, 2.2, 4.2', 6578: '2.1, 2.2, 3.1', 5848: '5', 5915: '5', 8710: '1.1, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.3', 9292: '5', 6805: '2.1, 2.2, 3.1, 3.3', 7281: '1.1, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2', 8728: '3.1, 3.3', 8333: '2.1, 2.2, 3.4', 4558: '2.1, 4.1', 7232: '1.1, 1.2, 2.1, 2.2, 3.1, 3.3, 4.1, 4.2', 8894: '1.1, 2.1, 3.1', 4240: '5', 6425: '4.1, 4.3', 7435: '2.1, 2.2, 3.1, 4.1', 9318: '2.2', 8968: '2.1, 2.2', 5812: '5', 6072: '5', 6290: '5', 7827: '2.1, 2.2', 4258: '2.1, 2.2, 3.4', 5944: '5', 8397: '1.1, 1.2, 2.1, 2.2, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2', 8641: '2.1', 9218: '5', 7226: '2.2', 8233: '2.1, 4.2', 8777: '2.1, 2.2, 3.1, 3.3, 4.2', 5368: '5', 5031: '5', 8611: '1.1', 9552: '2.1, 3.1', 9126: '5', 8196: '1.1', 7959: '2.1, 2.2, 3.1, 4.3', 4859: '5', 4356: '1.1, 2.1, 4.2', 4605: '5', 8